<a href="https://colab.research.google.com/github/NicolasHinke/python_sis_26_2/blob/main/aula_11/modelo_atividade_aula11_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Aula 11
Extrator de Dados Fiscais: XML para Excel
Você integra a equipe de Governança e Automação de TI. O departamento fiscal recebe arquivos XML de Notas Fiscais Eletrônicas e precisa consolidar os dados das compras da empresa em uma única planilha Excel (.xlsx) para auditoria de pagamentos.
Requisitos:  

Pesquise sobre leitura/escrita de XML em Python e crie scripts em Python que executem as etapas a seguir:

Etapa 1: Importações e Padrão PEP 8.

Importe os módulos nativos, de terceiros e essenciais para seu uso. mantendo a separação recomendada pela PEP 8.

In [1]:
import os
import csv
import xml.etree.ElementTree as ET
from pathlib import Path

Etapa 2: Coleta e Varredura de XMLs.

Crie uma pasta com pelo menos 3 arquivos .xml de notas fiscais (ou utilize amostras públicas/simuladas de NF-e). Seu script deve percorrer a pasta lendo arquivo por arquivo.

In [2]:
# Script para criar 3 notas fiscais simuladas no Colab:
import os

os.makedirs("notas_xml", exist_ok=True)

notas_amostra = [
    {"num": "1001", "data": "2026-10-10", "fornecedor": "Tech Solutions LTDA", "cnpj": "12.345.678/0001-90", "total": "4500.00"},
    {"num": "1002", "data": "2026-10-12", "fornecedor": "Cloud Cloud Servicos", "cnpj": "98.765.432/0001-11", "total": "1280.50"},
    {"num": "1003", "data": "2026-10-15", "fornecedor": "Hardware & Redes SA", "cnpj": "45.123.890/0001-55", "total": "8900.00"}
]

for n in notas_amostra:
    conteudo = f"""<?xml version="1.0" encoding="UTF-8"?>
<nfeProc xmlns="http://www.portalfiscal.inf.br/nfe">
    <NFe>
        <infNFe>
            <ide>
                <nNF>{n['num']}</nNF>
                <dhEmi>{n['data']}T14:30:00-03:00</dhEmi>
            </ide>
            <emit>
                <CNPJ>{n['cnpj']}</CNPJ>
                <xNome>{n['fornecedor']}</xNome>
            </emit>
            <total>
                <ICMSTot>
                    <vNF>{n['total']}</vNF>
                </ICMSTot>
            </total>
        </infNFe>
    </NFe>
</nfeProc>"""
    with open(f"notas_xml/nfe_{n['num']}.xml", "w", encoding="utf-8") as f:
        f.write(conteudo)

print("Amostras criadas na pasta 'notas_xml'!")


Amostras criadas na pasta 'notas_xml'!


In [3]:
pasta_xml = Path("notas_xml")

arquivos_xml = list(pasta_xml.glob("*.xml"))

print(f"Total de arquivos XML encontrados: {len(arquivos_xml)}")

for arquivo in arquivos_xml:
    print(arquivo.name)

Total de arquivos XML encontrados: 3
nfe_1001.xml
nfe_1003.xml
nfe_1002.xml


Etapa 3: Extração

Para cada nota fiscal encontrada, extraia com tratamento de erro (try/except) pelo menos as seguintes informações:
- Número da Nota
- Data de Emissão
- Nome do Fornecedor / Emitente
- CNPJ do Emitente
- Valor Total da Nota

In [4]:
dados_notas = []

def buscar_valor(elemento, nome_tag):
    for item in elemento.iter():
        if item.tag.split("}")[-1] == nome_tag:
            return item.text
    return ""

for arquivo in arquivos_xml:
    try:
        arvore = ET.parse(arquivo)
        raiz = arvore.getroot()

        numero = buscar_valor(raiz, "nNF")
        data = buscar_valor(raiz, "dhEmi")
        fornecedor = buscar_valor(raiz, "xNome")
        cnpj = buscar_valor(raiz, "CNPJ")
        valor = buscar_valor(raiz, "vNF")

        dados_notas.append([
            numero,
            data,
            fornecedor,
            cnpj,
            float(valor)
        ])

    except (ET.ParseError, ValueError, OSError) as erro:
        print(f"Erro ao processar {arquivo.name}: {erro}")

print("Extração concluída!")

for nota in dados_notas:
    print(nota)

Erro ao processar nfe_1003.xml: not well-formed (invalid token): line 11, column 33
Extração concluída!
['1001', '2026-10-10T14:30:00-03:00', 'Tech Solutions LTDA', '12.345.678/0001-90', 4500.0]
['1002', '2026-10-12T14:30:00-03:00', 'Cloud Cloud Servicos', '98.765.432/0001-11', 1280.5]


Etapa 4: Geração da Planilha.

Crie um arquivo csv ou um arquivo xlsx.
Insira as linhas coletadas e formate o valor monetário com duas casas decimais (:.2f).
Salve o arquivo .

In [5]:
with open(
    "notas_fiscais_consolidadas.csv",
    "w",
    newline="",
    encoding="utf-8-sig"
) as arquivo:

    escritor = csv.writer(arquivo, delimiter=";")

    escritor.writerow([
        "Número da Nota",
        "Data de Emissão",
        "Fornecedor",
        "CNPJ",
        "Valor Total"
    ])

    for nota in dados_notas:
        escritor.writerow([
            nota[0],
            nota[1],
            nota[2],
            nota[3],
            f"{nota[4]:.2f}"
        ])

print("Planilha CSV gerada com sucesso!")
print(f"Total de notas exportadas: {len(dados_notas)}")

Planilha CSV gerada com sucesso!
Total de notas exportadas: 2
